# LeafLiteNet: training on Google Colab (GPU)

Trains our lightweight CNN **LeafLiteNet** (`src/model/lite_cnn.py`, about 1.3M parameters) from scratch on the
same **leaf-grouped train/val/test split** as the ResNet50 model, then evaluates it with the same metrics.

| Run | What it is | Purpose in the paper |
|---|---|---|
| `leaflitenet` | full model | main result |
| `leaflitenet_noSE` | without squeeze-and-excitation | ablation: does channel attention help? |
| `leaflitenet_w0.5` | half the channels (0.36M parameters) | ablation: size vs accuracy |

**Before you start:** upload `Dataset_raw.zip` (made on the laptop, about 960 MB) to Google Drive in a folder
named `SmartCropCare` (so the file is `MyDrive/SmartCropCare/Dataset_raw.zip`).

**If Colab disconnects:** run the setup cells (1 to 4) again, then the same training cell. Training resumes from the
last finished epoch, because progress is saved to Google Drive.

## 1. Use a GPU
Menu **Runtime → Change runtime type → T4 GPU**, then run this cell. It must list a GPU.

In [3]:
!nvidia-smi -L
import tensorflow as tf
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))
assert tf.config.list_physical_devices("GPU"), "No GPU: set Runtime -> Change runtime type -> T4 GPU"

GPU 0: Tesla T4 (UUID: GPU-f824e181-5bf1-760f-f3f9-7be4c1c49642)
TensorFlow 2.21.0 | GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Connect Google Drive (results are saved there)

In [4]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

DRIVE = "/content/drive/MyDrive/SmartCropCare"
RUNS = f"{DRIVE}/runs"            # trained models and reports are written here
ZIP = f"{DRIVE}/Dataset_raw.zip"
import os
assert os.path.exists(ZIP), f"Upload Dataset_raw.zip to {DRIVE} first"
print("Dataset zip found:", round(os.path.getsize(ZIP) / 1e6), "MB")

Mounted at /content/drive
Dataset zip found: 872 MB


## 3. Get the project code

In [5]:
%cd /content
# The ResNet50 model file is stored with Git LFS and is not needed here, so skip downloading it.
!GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 https://github.com/JemisKevadiya/Smart-Crop-Care-System-.git scc 2>/dev/null || (cd scc && git pull)
%cd /content/scc
!git log --oneline -1
assert os.path.exists("src/model/lite_cnn.py"), "lite_cnn.py missing: push the latest code to GitHub first"

/content
/content/scc
20b7995 (grafted, HEAD -> main, origin/main, origin/HEAD) Created using Colab


## 4. Unpack the dataset and check every image of the split is present

In [6]:
!unzip -q -o "{ZIP}" -d /content/scc
import pandas as pd
missing = [p for s in ("train", "val", "test") for p in pd.read_csv(f"data/splits/{s}.csv")["path"]
           if not os.path.exists(p)]
print("missing images:", len(missing))
assert not missing, missing[:5]

missing images: 0


## 5. Quick check (about 1 minute)
Trains 3 tiny steps and evaluates 64 images, only to confirm everything runs. Accuracy here is meaningless.

In [7]:
!python scripts/train_lite_cnn.py --smoke --root /content/smoke_test

W0000 00:00:1791356357.924247    1754 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
leaflitenet_smoke: 1,291,292 parameters, 194M MACs, starting
Epoch 1/2

val_accuracy improved to 0.0000; saved /content/smoke_test/models/lite_cnn/leaflitenet_smoke/best.keras
3/3 - 35s - 12s/step - accuracy: 0.0000e+00 - loss: 3.8250 - top3: 0.0417 - val_accuracy: 0.0000e+00 - val_loss: 3.6346 - val_top3: 0.0000e+00 - learning_rate: 0.0010
Epoch 2/2
3/3 - 1s - 393ms/step - accuracy: 0.0000e+00 - loss: 3.8638 - top3: 0.0833 - val_accuracy: 0.0000e+00 - val_loss: 3.6384 - val_top3: 0.0000e+00 - learning_rate: 0.0010
1/1 - 11s - 11s/step
train {"images": 64, "accuracy": 0.0, "top3_accuracy": 0.0, "loss": 3.6346113681793213, "precision_macro": 0.0, "recall_macro": 0.0, "f1_macro": 0.0, "precision_weighted": 0.0, "recall_weighted": 0.0, "f1_weighted": 0.0}
1/1 - 1s - 611ms/step
val {"images": 64, "ac

## 6. Train the main model
Up to 40 epochs; early stopping ends it when validation loss stops improving.
The first epoch is slower (it decodes and caches all images); later epochs are faster.
Then the model is evaluated on train, validation and test.

In [ ]:
!python scripts/train_lite_cnn.py --run leaflitenet --root "{RUNS}" --cache-dir /content/cache

W0000 00:00:1791356423.248451    2335 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
leaflitenet: 1,291,292 parameters, 194M MACs, starting
Epoch 1/40


## 7. Ablation runs (for the paper)
Run these after step 6 (one at a time). Each is a full training run.

In [ ]:
!python scripts/train_lite_cnn.py --run leaflitenet_noSE --no-se --root "{RUNS}" --cache-dir /content/cache

In [ ]:
!python scripts/train_lite_cnn.py --run leaflitenet_w0.5 --width 0.5 --root "{RUNS}" --cache-dir /content/cache

## 8. Results table

In [ ]:
import json, glob
rows = []
for f in sorted(glob.glob(f"{RUNS}/artifacts/reports/lite_cnn/*/model_metrics.json")):
    m = json.load(open(f)); t = m["evaluation"]["test"]
    rows.append({"run": f.split("/")[-2], "params (M)": round(m["params"] / 1e6, 2),
                 "MACs (M)": round(m["macs"] / 1e6), "size (MB)": m["file_size_mb"],
                 "epochs": m["epochs_trained"], "test acc": round(t["accuracy"], 4),
                 "macro F1": round(t["f1_macro"], 4), "top-3": round(t["top3_accuracy"], 4),
                 "val acc": round(m["evaluation"]["val"]["accuracy"], 4)})
rows.append({"run": "ResNet50 (reference, trained earlier)", "params (M)": 24.12, "MACs (M)": 3857,
             "test acc": 0.9762, "macro F1": 0.9761, "top-3": 0.9992, "val acc": 0.9772})
pd.DataFrame(rows)

## 9. Bring the results back to the laptop
In Google Drive, download the folder `SmartCropCare/runs`. Copy its `models/lite_cnn` and
`artifacts/reports/lite_cnn` folders into the project folder (same paths). The CPU speed comparison for the paper
(`scripts/compare_models.py`) is then run on the laptop, so both models are timed on the same machine.